# Časť 4, Téma 1: Vzťah medzi spotrebou energie a Hammingovou váhou

---
POZNÁMKA: Toto laboratórne cvičenie odkazuje na niektoré (komerčné) školiace materiály na stránke [ChipWhisperer.io](https://www.ChipWhisperer.io). Lab môžete voľne spúšťať a používať v súlade s open-source licenciou (vrátane použitia vo vašich vlastných kurzoch, ak ich budete distribuovať podobným spôsobom), ale musíte zachovať informáciu o tomto pôvodnom umiestnení. Ak si chcete užiť plnohodnotný zážitok, zvážte účasť na našom školení.

---

**ZHRNUTIE:** *V predchádzajúcej časti kurzu SCA101 sme využili skutočnosť, že existuje vzťah medzi hodnotou bitov, s ktorými mikrokontrolér manipuluje, a spotrebou energie, aby sme získali kľúč AES.*

*V tomto labe náš model vylepšíme tým, že sa pozrieme na to, ako spotreba energie súvisí s Hammingovou váhou.*

**CIELE VZDELÁVANIA:**

* Pochopiť, čo je to Hammingova váha
* Vypočítať Hammingovu váhu jedného bajtu
* Identifikovať bod, v ktorom sa zapisuje výstup z SBoxu
* Vykresliť graf spotreby energie na výstupe z SBoxu v závislosti od Hammingovej váhy


## Model AES

Opäť sa pozrieme na SBox, takže si vezmite svoju implementáciu z minulej časti:

In [ ]:
# ###################
# Add your code here
# ###################
raise NotImplementedError("Add your code here, and delete this.")

<details>
<summary>Nejaké riešenie</summary>
<pre><code>
    sbox = [
    # 0    1    2    3    4    5    6    7    8    9    a    b    c    d    e    f 
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76, # 0
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0, # 1
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15, # 2
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75, # 3
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84, # 4
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf, # 5
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8, # 6
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2, # 7
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73, # 8
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb, # 9
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79, # a
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08, # b
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a, # c
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e, # d
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf, # e
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16  # f
]

def aes_internal(inputdata, key):
    return sbox[inputdata ^ key]
</code></pre>
</details>

In [ ]:
#Simple test vectors - if you get the check-mark printed all OK.
assert(aes_internal(0xAB, 0xEF) == 0x1B)
assert(aes_internal(0x22, 0x01) == 0x26)
print("✔️ OK to continue!")

## Hammingova váha

Pripomeňme si, že dôvodom, prečo existuje vzťah medzi spotrebou energie a internými dátami mikrokontroléra, je to, že nastavovanie týchto dát vyžaduje energiu. Taktiež sme videli, že čím viac dát sa nastavuje, tým väčší je priemerný odber prúdu. Nie je preto prekvapivé, že by mal existovať nejaký konzistentný vzťah medzi počtom bitov nastavených na hodnotu 1 (čo nazývame **Hammingova váha**) a energiou spotrebovanou pri tomto procese.

Hammingova váha, napriek tomu, že ide o pomerne jednoduchú myšlienku, v skutočnosti nie je úplne triviálna na výpočet (pozri https://en.wikipedia.org/wiki/Hamming_weight#Efficient_implementation). Mohli by ste na to napísať vlastnú funkciu, ale v Pythone je oveľa jednoduchšie proste previesť číslo na reťazec bitov a spočítať znaky `"1"`:

In [ ]:
def calc_hamming_weight(n):
    return bin(n).count("1")

Ešte lepšie bude vytvoriť si vyhľadávaciu tabuľku (lookup table) – t. j. urobiť výpočet pre každé číslo od 0 do 255 a uložiť ich do poľa:

In [ ]:
# ###################
# Add your code here
# ###################
raise NotImplementedError("Add Your Code Here")

In [ ]:
assert HW[0x53] == 4
print("✔️ OK to continue!")

Prvým problémom, na ktorý narazíme, je, že nevieme, kde presne operácia SBox prebieha. Mala by sa vykonávať niekde blízko začiatku (skúsme odhadnúť, že v rámci prvých 2000 vzoriek). Jednou z možností je, že by sme mohli rozdeliť výkonové krivky (traces) do skupín podľa Hammingovej váhy a každej priradiť farbu. Ak to vykreslíme, možno sa nám podarí nájsť nejaký vzor:

In [ ]:
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.palettes import brewer

output_notebook()
p = figure()

plot_start = 0
plot_end = 2000
xrange = list(range(len(trace_array[0]))[plot_start:plot_end])
bnum = 0
color_mapper = brewer['PRGn'][9]

for tnum in range(len(trace_array)):
    hw_of_byte = HW[aes_internal(textin_array[tnum][bnum], key[bnum])]
    p.line(xrange, trace_array[tnum][plot_start:plot_end], line_color=color_mapper[hw_of_byte])

show(p)

Nanešťastie pravdepodobne zistíte, že tento graf vám toho veľa nepovie; tá časť spotreby energie, ktorá súvisí s výstupom z SBoxu, je jednoducho príliš malá na to, aby sa dala jasne rozoznať. Mohli by sme skúsiť spriemerovať jednotlivé skupiny Hammingových váh, aby boli rozdiely zreteľnejšie, no to nerieši zásadný problém, že výstup z SBoxu sa stráca v šume všetkého ostatného, čo sa na čipe deje.

Namiesto toho sa na to skúsme pozrieť z iného uhla. V skutočnosti chceme odstrániť celkový „tvar“ krivky a ponechať iba signál pochádzajúci z výstupu SBoxu. Mohli by sme proste vybrať jednu krivku a odpočítať ju od každej skupiny, ale odčítanie priemeru *všetkých* kriviek urobí graf oveľa prehľadnejším. Ešte lepšie by bolo mať rovnomerné zastúpenie medzi všetkými skupinami Hammingových váh, keďže extrémne Hammingove váhy (0 a 8) sú oveľa vzácnejšie než tie stredné hodnoty, avšak to nakoniec nebude úplne nutné (hoci ak chcete, môžete sa o to pokúsiť). Graf bude tiež prehľadnejší (a vykreslí sa oveľa rýchlejšie), ak spriemerujeme všetky skupiny Hammingových váh, čím eliminujeme prípadné anomálie. Skúste to implementovať (samotné vykreslenie grafu opäť vyriešime za vás):

In [ ]:
# ###################
# Add your code here
# ###################
raise NotImplementedError("Add Your Code Here")

xrange = range(len(trace_array[0]))[plot_start:plot_end]
color_mapper = brewer['PRGn'][9]
for hw in range(9):
    p.line(xrange, (hw_averages[hw]-avg_trace)[plot_start:plot_end], line_color=color_mapper[hw])
    
show(p)

Po vykreslení tohto grafu by ste mali uvidieť jedno veľmi jasné miesto, kde sa farby od seba oddeľujú. Presne tam prebieha operácia SBox. V skutočnosti je to pravdepodobne natoľko zreteľné, že pozíciu SBoxu môžete určiť výhradne podľa miesta, kde má graf najväčšiu hodnotu:

In [ ]:
sbox_loc = np.argmax(abs(hw_averages[0]-avg_trace))

Teraz, keď už vieme, kde presne k operácii SBox dochádza, vykreslite priemerné hodnoty pre jednotlivé Hammingove váhy v bode `sbox_loc` v závislosti od ich Hammingovej váhy.

**POMÔCKA: Možno bude fajn previesť vaše `hw_averages` na numpy pole, aby ste k nemu mohli pristupovať po stĺpcoch. Zápis `hw_averages[:,sbox_loc]` vám vráti hodnoty `hw_averages` v bode `sbox_loc`.**

In [ ]:
# ###################
# Add your code here
# ###################
raise NotImplementedError("Add Your Code Here")

Mali by ste zistiť, že tento vzťah je prevažne lineárny, čo pre vás zrejme nebude žiadnym veľkým prekvapením. Dáva predsa zmysel, že nastavenie 8 dátových bitov spotrebuje zhruba 8-krát viac energie než nastavenie jedného bitu.

Pravdepodobne tiež zistíte, že sklon (smernica) tohto vzťahu je záporný – pokiaľ teda nepracujete s ChipWhisperer Nano. Má to svoj dobrý dôvod. Ak si spomínate, ako meriame prúd tečúci do zariadenia, zistíte, že napätie KLESÁ pri ZVÝŠENÍ prúdu. Môžete to vidieť na nasledujúcom diagrame:

```
        Rshunt
(Vin)----v^v^v^----------||------(Do ChipWhisperer)
                |
                |
       (Do mikrokontroléra)
```

V prípade ChipWhisperer Nano je sklon kladný kvôli prítomnosti invertujúceho zosilňovača na vstupe meracieho portu.

Meriame úbytok napätia na bočníku (meracom odpore - shunt resistor). Zvýšenie prúdu spôsobí vyššie napätie na odpore. Keď nepreteká žiadny prúd, nevzniká na odpore žiadny úbytok. Keďže však meriame iba jeden koniec odporu, vidíme vyššie napätie vtedy, keď žiadny prúd netečie.

Tento sklon môžeme napraviť jednoduchým otočením smeru merania (pridaním znamienka mínus `-` pred nameranú hodnotu).

Teraz, keď už viete, kde sa operácia SBox odohráva, skúste sa vrátiť k pôvodnému grafu a priblížiť (zoomnúť) si túto konkrétnu sekciu. Dokážete už teraz rozoznať rozdiely medzi jednotlivými Hammingovými váhami?

## Závery a ďalšie kroky

Vďaka tomuto labu by ste mali byť dostatočne presvedčení o tom, že existuje lineárny vzťah medzi Hammingovou váhou dát zapisovaných v mikrokontroléri a energiou, ktorú pri tom spotrebováva.

V nasledujúcom labe uvidíme, ako sa dá táto skutočnosť využiť na výrazné vylepšenie nášho pôvodného DPA útoku.

---
<small>UPOZORNENIE: Tento materiál je chránený autorskými právami (C) NewAE Technology Inc., 2015–2020. ChipWhisperer je ochranná známka spoločnosti NewAE Technology Inc., na ktorú si táto spoločnosť uplatňuje práva vo všetkých jurisdikciách a ktorá je registrovaná minimálne v Spojených štátoch amerických, Európskej únii a Čínskej ľudovej republike.

Návody odvodené z našej open-source práce musia byť uverejnené pod príslušnou open-source licenciou a upozornenie na zdroj musí byť *jasne zobrazené*. Iba pôvodní držitelia autorských práv môžu udeľovať licencie alebo povoľovať ďalšiu distribúciu – hoci spoločnosť NewAE Technology Inc. vlastní autorské práva k mnohým návodom, repozitár github obsahuje príspevky komunity, ktoré nemôže licencovať za špeciálnych podmienok a **musia** byť udržiavané ako open-source verzia. V prípade potreby prosím kontaktujte NewAE Technology Inc ohľadom špeciálnych povolení (ak je to možné)

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.</small>